# Running inference on the accelerator

<centering>
<img src="./images/pynq-z2.jpeg" width="300" />
</centering>

**Important**: this section runs on the pynq-z2 board

In this section we will use the FPGA bitfile that we produced in part 1 to make predictions on the pynq-z2 board. This System-on-Chip device has an ARM CPU (or PS / Processing System in the SoC terminology) that is running an Operating System serving Jupyter Notebooks. The CPU is able to communicate with the FPGA (or PL / Programmable Logic in the SoC terminology). Our BDT bitfile will be loaded onto the FPGA part of the device.

`conifer` provides a runtime driver using AMD's `pynq` library providing Python APIs to interact with the FPGA.

<centering>
<img src="./images/zynq_block_diagram.jpg" width="600" />
</centering>

In [ ]:
# upload the bitfile built on another machine and unzip it
!unzip conifer_xgboost_moons.zip

In [ ]:
import conifer
import numpy as np
import json

print(f'Using conifer version {conifer.__version__}')

## Load the bitfile
This step creates a `ZynqDriver` object, loading the bitfile onto the PL in the process. We also initialise some buffers that will be used to transfer data between the host (PS) and accelerator (PL). The buffers are _memory mapped_ to the PL. There is analagously an `AlveoDriver` for Alveo devices.

We set the batch size here to the size of the data that we will run on. If you don't know what it should be at the moment you create the accelerator, you can set it later instead with `accelerator._init_buffers(batch_size)`.

In [ ]:
accelerator = conifer.backends.xilinxhls.runtime.ZynqDriver('conifer_xgboost_moons.bit', batch_size=1000*1000)

In [ ]:
# print the docs of the `ZynqDriver`
help(accelerator)

In [ ]:
print(f'Number of features: {accelerator.n_features}')
print(f'Number of classes:  {accelerator.n_classes}')

## Prepare data

We'll make the same decision boundary plot that we used to visualise the model already. So we make a regular 1000x1000 grid of points.

Note we explicitly set the data type to `float32` to match the `InterfaceType: float` setting in the configuration used to build the bitfile.

In [ ]:
# make a 1000x1000 grid of points in the feature space
X_mesh = np.meshgrid(np.linspace(-3, 3, 1000), np.linspace(-3, 3, 1000))
# reshape them for inference
X_grid = np.vstack([X_mesh[0].ravel(), X_mesh[1].ravel()]).T.astype(np.float32)

## Run inference

This command runs inference of the BDT on the PL by copying the array into the memory mapped input buffer, prompting the BDT IP to start processing, and copying the contents of the output buffer when it signals that it's finished.

In [ ]:
y_pynq = accelerator.decision_function(X_grid)

In [ ]:
%%timeit
y_pynq = accelerator.decision_function(X_grid)

In [ ]:
# just print a few examples
# they will probably be quite similar due to the grid we're using for predictions
print(y_pynq[:10])

## Save

We won't do more analysis of the results on the pynq-z2 itself, we'll copy the data back to the desktop PC and take a look at it there.

In [ ]:
np.save('y_mesh_pynq_hls.npy', y_pynq)

# Forest Processing Unit

Now we'll run inference of the same model on the `conifer` FPU. This is the reconfigurable DF accelerator

First we need to download it.

In [ ]:
!wget https://ssummers.web.cern.ch/conifer/downloads/v1.5/pynq-z2/fpu_100TE_512N_16F_16T_16S_DS/fpu.zip
!unzip fpu.zip

## Load FPU onto FPGA

We start by downloading the FPU bitfile onto the PL. The FPU isn't initially loaded with any specific model, we'll load the model in a second step.

In [ ]:
fpu_device = conifer.backends.fpu.runtime.ZynqDriver('fpu.bit', batch_size=1)

## Inspect FPU

The FPU bitfile holds internally the configuration settings that it was built with, so we can read them now.

In [ ]:
# print the FPU config, read from the device
print('Modified Configuration\n' + '-' * 50)
print(json.dumps(fpu_device.config, indent=2))
print('-' * 50)

## Load model

Now we load the specific BDT we want to work with. The `conifer` FPU runtime will convert it into data that can be loaded onto the FPU device. When we convert the model, we have to specify a configuration that includes the details of the device we will load the model onto. In this case we retrieve that configuration from the device itself, but it could also come from a JSON file. At this step we don't actually download the model onto the device.

In [ ]:
cfg = conifer.backends.fpu.auto_config()
cfg['FPU'] = fpu_device.config
fpu_model = conifer.model.load_model('conifer_xgboost_moons.json', new_config=cfg)

# this is a workaround for the not-quite working dynamic scaling in the FPU
 
fpu_model.scale(1000 / fpu_model.threshold_scale, 1000 * fpu_model.score_scale[0])
fpu_model.threshold_scale = 1000
fpu_model.score_scale = 1./1000

In [ ]:
fpu_model.threshold_scale, fpu_model.score_scale

## Attach device

Now we send the model to the FPU, ready for inference. Again we specify the size of buffers to allocate.

In [ ]:
fpu_model.attach_device(fpu_device, batch_size=1000*1000)

## Run inference

Now we make predictions on the FPU with the same mesh of points, and save them.

In [ ]:
y_pynq_fpu = fpu_model.decision_function(X_grid)

In [ ]:
%%timeit
fpu_model.decision_function(X_grid)

In [ ]:
# just print a few examples
# they will probably be quite similar due to the grid we're using for predictions
print(y_pynq_fpu[:10])

In [ ]:
np.save('y_mesh_pynq_fpu.npy', y_pynq_fpu)